CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)
This notebook contains exactly 3 sections:

Infrastructure Sensing & World Transformation (Theory, camera setups, and 2D-to-3D projection)

V2X Communication & Message Serialization (Asynchronous message design, delay simulation, and protocol payloads)

End-to-End System Integration & Occlusion Scenarios (The runnable project execution with full evaluation metrics)

CARLA docs
Main docs: https://carla.readthedocs.io/en/latest/

Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/

Python API: https://carla.readthedocs.io/en/latest/python_api/

In [ ]:
import carla
import time
import random
import cv2
import queue
import threading
import json
import math
import numpy as np
from datetime import datetime

In [ ]:
# Initialize client and connect to the CARLA server
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)
world = client.get_world()
spectator = world.get_spectator()
blueprint_library = world.get_blueprint_library()

In [ ]:
def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

## 1. RGB Camera: how it works + configurable parameters

`spawn_camera(world, attach_to, transform, width=640, height=360, fov=95, tick=0.05)` mounts an RGB sensor on a parent actor.

What each argument controls:
- `world`: CARLA world used to fetch blueprint and spawn the camera actor.
- `attach_to`: parent actor (usually the ego vehicle).
- `transform`: camera mount pose relative to parent (`x, y, z, pitch, yaw, roll`).
- `width` -> `image_size_x`: image width in pixels.
- `height` -> `image_size_y`: image height in pixels.
- `fov`: horizontal field of view in degrees (smaller = zoom-like, larger = wide-angle).
- `tick` -> `sensor_tick`: seconds between frames (`0.05` ~= 20 FPS max).

Data pipeline in callback:
1. CARLA provides `carla.Image`.
2. `raw_data` is BGRA bytes.
3. Convert to OpenCV BGR with `image_to_bgr`.

Camera parameters you should tune first in practice:
- Resolution (`width`, `height`): visual detail vs compute cost.
- `fov`: scene coverage vs geometric distortion.
- `sensor_tick`: temporal smoothness vs processing load.
- Mount `transform`: what the model can actually see.

Blueprint-level camera attributes can include many additional options (for example post-processing, exposure, bloom, lens effects) depending on CARLA version.
Run the next cell to inspect the exact attributes available in your installation.


### Camera parameter reference from your installed CARLA build

Run the next cell to print all camera blueprint attributes, current value, mutability, and recommended values.


In [ ]:

def spawn_camera(world, attach_to, transform, width=640, height=360, fov=95, tick=0.05):
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    return world.spawn_actor(bp, transform, attach_to=attach_to)

def image_to_bgr(image):
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()



In [ ]:
# Camera blueprint attributes (compatible across CARLA versions)
bp = world.get_blueprint_library().find("sensor.camera.rgb")
print("camera blueprint id:", bp.id)


def _attr_value(attr):
    # Some CARLA builds expose default_value, others do not.
    for name in ("default_value", "value", "default"):
        if hasattr(attr, name):
            try:
                v = getattr(attr, name)
                if v is not None:
                    return str(v)
            except Exception:
                pass
    try:
        return str(attr)
    except Exception:
        return "<n/a>"


for attr in bp:
    attr_id = getattr(attr, "id", "<unknown>")
    attr_type = str(getattr(attr, "type", ""))
    modifiable = bool(getattr(attr, "is_modifiable", False))
    rec = [str(v) for v in list(getattr(attr, "recommended_values", []))]
    rec_show = ", ".join(rec[:8]) + (" ..." if len(rec) > 8 else "")
    value = _attr_value(attr)
    print(f"{attr_id:26} type={attr_type:12} value={value:>10} modifiable={modifiable} rec=[{rec_show}]")


1. Infrastructure Sensing & World TransformationMathematical Foundation: 2D Image Space to 3D World SpaceTo send meaningful target tracking coordinates to an Ego vehicle, a static roadside infrastructure camera must map a detected object's pixel coordinate $(u, v)$ back into a 3D World coordinate $(X_w, Y_w, Z_w)$.This is achieved using the camera Intrinsic Matrix ($K$) and Extrinsic Matrix ($[R|t]$):$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$Given that the ground plane can be structurally approximated ($Z_w \approx 0$), we solve for the scaling factor $\lambda$ to project inverse coordinates from the pixel plane back through the inverted rigid transformation matrix of the camera mount location.Configurable Parameters GuideCamera Extrinsics: Positioning height ($z \ge 6.0\text{m}$) and dramatic downward pitch ($\text{pitch} \approx -35^\circ$) are vital to eliminate self-occlusion artifacts within blind intersections.Intrinsic Matrix Fields: Tuning Resolution ($W, H$) and Horizontal Field of View ($\text{FOV}$) determines pixel density per meter at long range.

def get_camera_intrinsic_matrix(width, height, fov):
    """Computes the intrinsic matrix K for a pinhole camera model."""
    focal = width / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = focal
    K[1, 1] = focal
    K[0, 2] = width / 2.0
    K[1, 2] = height / 2.0
    return K

def spawn_roadside_camera(world, transform, width=800, height=600, fov=90, tick=0.05):
    """Spawns a static infrastructure sensor frame."""
    bp = world.get_blueprint_library().find("sensor.camera.rgb")
    bp.set_attribute("image_size_x", str(width))
    bp.set_attribute("image_size_y", str(height))
    bp.set_attribute("fov", str(fov))
    bp.set_attribute("sensor_tick", str(tick))
    
    # Static infrastructure elements do not use an attach_to actor
    return world.spawn_actor(bp, transform)

def image_to_bgr(image):
    """Converts CARLA raw image array to standard BGR layout."""
    arr = np.frombuffer(image.raw_data, dtype=np.uint8)
    arr = np.reshape(arr, (image.height, image.width, 4))
    return arr[:, :, :3].copy()

2. V2X Communication & Message Serialization
Asynchronous I2V V2X Message Design
Real-world infrastructure messaging (such as ETSI ITS-G5 or C-V2X) relies on standardized Cooperative Awareness Messages (CAM) or Collective Perception Messages (CPM). In this implementation, the infrastructure packages objects using an asymmetric JSON schema over an emulated message broker queue.

V2X Data Payload Spec (JSON String)

In [ ]:
{
  "timestamp": 1716885368.123,
  "station_id": 9901,
  "detected_objects": [
    {
      "class": "pedestrian",
      "world_x": 124.52,
      "world_y": -45.12,
      "world_z": 0.05,
      "confidence": 0.94
    }
  ]
}

Network Latency & Synchronization TheoryWireless transmission incurs structural degradation over distance alongside discrete propagation delays. Our simulation architecture models this via a thread-safe broker loop that explicitly injects bounded latency ($\Delta t \in [20\text{ms}, 100\text{ms}]$) before messages arrive at the Ego vehicle's ingestion layer.

In [ ]:
class V2XMessageBroker:
    """Simulates an asynchronous network broker with configurable latency."""
    def __init__(self, latency_ms=50):
        self.queue = queue.Queue()
        self.latency_seconds = latency_ms / 1000.0
        self._lock = threading.Lock()
        
    def publish(self, payload_dict):
        """Published by Infrastructure side."""
        transmission_time = time.time() + self.latency_seconds
        self.queue.put((transmission_time, json.dumps(payload_dict)))
        
    def receive_latest(self):
        """Consumed by Ego Vehicle side."""
        now = time.time()
        latest_valid_msg = None
        
        # Pull all messages that have cleared the simulated flight latency window
        temp_list = []
        while not self.queue.empty():
            try:
                item = self.queue.get_nowait()
                if now >= item[0]:
                    latest_valid_msg = item[1]
                else:
                    temp_list.append(item)
            except queue.Empty:
                break
                
        # Return pending delayed packets back to the queue
        for pending_item in temp_list:
            self.queue.put(pending_item)
            
        if latest_valid_msg:
            return json.loads(latest_valid_msg)
        return None

3. End-to-End System Integration & Occlusion Scenarios
We will now build a comprehensive evaluation scenario. An infrastructure camera is placed looking over a blind corner where a large building blocks the local view of an approaching Ego vehicle. A pedestrian is routed to cross directly into the vehicle's path.

Local Perception vs Shared Infrastructure Fusion
Without V2X Assistance: The Ego vehicle relies solely on its front bumper line-of-sight sensors, resulting in emergency braking and a high collision rate.

With V2X Assistance: The Ego vehicle continuously consumes the message broker queue, projects the shared target location into its local coordinate system, and computes early proactive speed reduction profiles.

In [ ]:
def infrastructure_processing_loop(cam_sensor, broker, stop_event, camera_transform, width=800, height=600, fov=90):
    """
    Simulates the infrastructure node. Ingests raw camera frames, isolates target
    actors through simulated 2D detection, and issues V2X telemetry.
    """
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)
    
    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            bgr_frame = image_to_bgr(image)
            
            # Context querying to simulate ground-truth tracking transformation mapping
            world_context = cam_sensor.get_world()
            actors = world_context.get_actors()
            pedestrians = actors.filter("walker.pedestrian.*")
            cyclists = actors.filter("vehicle.bpc.bicycle")
            vulnerable_road_users = list(pedestrians) + list(cyclists)
            
            detected_payloads = []
            
            for vru in vulnerable_road_users:
                vru_loc = vru.get_transform().location
                # Validate distance profile relative to static station limits (e.g., 45 meters)
                if cam_sensor.get_transform().location.distance(vru_loc) < 45.0:
                    detected_payloads.append({
                        "class": "pedestrian" if "walker" in vru.type_id else "cyclist",
                        "world_x": vru_loc.x,
                        "world_y": vru_loc.y,
                        "world_z": vru_loc.z,
                        "confidence": round(random.uniform(0.92, 0.99), 2)
                    })
            
            # Broadcast payload package across V2X interface
            if detected_payloads:
                msg = {
                    "timestamp": time.time(),
                    "station_id": 5501,
                    "detected_objects": detected_payloads
                }
                broker.publish(msg)
                
            # Optional: Render the Infrastructure perspective overlay
            cv2.putText(bgr_frame, f"V2X Node Active - Tracking: {len(detected_payloads)} VRUs", 
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imshow("Infrastructure Node Monitoring Feed", bgr_frame)
            cv2.waitKey(1)
            
        except queue.Empty:
            continue
            
    cam_sensor.stop()
    cv2.destroyWindow("Infrastructure Node Monitoring Feed")

Complete Evaluation Run (The Main Block)
Run this cell to launch the full occlusion scenario test loop. You can switch the USE_V2X_ASSISTANCE flag to compare how the vehicle behaves with and without collaborative assistance.

In [ ]:
# System Operational Flags
USE_V2X_ASSISTANCE = True  # Toggle to False to evaluate the unassisted occlusion control baseline
SCENARIO_DURATION = 15.0   # Operational timeout limit per run

# Define Actor Storage arrays
spawned_actors = []
network_stop_signal = threading.Event()
network_stop_signal.clear()

try:
    # 1. Setup Environment Maps and Spawning Points
    # Target intersection coordinates in default Town01 layout
    intersection_center = carla.Location(x=150.0, y=130.0, z=0.5)
    
    # Spawn Ego Vehicle
    ego_bp = blueprint_library.filter("vehicle.tesla.model3")[0]
    ego_tf = carla.Transform(carla.Location(x=120.0, y=132.0, z=1.0), carla.Rotation(yaw=0.0))
    ego_vehicle = world.spawn_actor(ego_bp, ego_tf)
    spawned_actors.append(ego_vehicle)
    
    # Spawn Vulnerable Road User (Crossing Pedestrian)
    ped_bp = random.choice(blueprint_library.filter("walker.pedestrian.*"))
    ped_tf = carla.Transform(carla.Location(x=152.0, y=142.0, z=1.5), carla.Rotation(yaw=-90.0))
    ped_actor = world.spawn_actor(ped_bp, ped_tf)
    spawned_actors.append(ped_actor)
    
    # Spawn Infrastructure Camera overlooking the intersection from high vantage point
    infra_tf = carla.Transform(carla.Location(x=155.0, y=125.0, z=9.5), carla.Rotation(pitch=-35.0, yaw=110.0))
    infra_camera = spawn_roadside_camera(world, infra_tf)
    spawned_actors.append(infra_camera)
    
    # 2. Instantiate V2X System Infrastructure Threads
    v2x_broker = V2XMessageBroker(latency_ms=40)
    infra_thread = threading.Thread(
        target=infrastructure_processing_loop, 
        args=(infra_camera, v2x_broker, network_stop_signal, infra_tf),
        daemon=True
    )
    infra_thread.start()
    
    # Initialize Metrics Logging
    min_distance_to_target = float("inf")
    v2x_lead_time_discovered = None
    collision_detected = False
    
    # Stabilize client sync ticks
    world.tick()
    time.sleep(0.5)
    
    # Start Moving the Pedestrian into the blind intersection area
    ped_control = carla.WalkerControl(direction=carla.Vector3D(0, -1, 0), speed=1.8)
    ped_actor.apply_control(ped_control)
    
    start_time = time.time()
    print(f">> Executing Occlusion Simulation Run. V2X Assistance Status: {USE_V2X_ASSISTANCE}")
    
    while time.time() - start_time < SCENARIO_DURATION:
        world.tick()
        
        # Update Viewer Spectator
        move_spectator_to(ego_vehicle.get_transform(), spectator, distance=14.0, z=5.0, pitch=-20.0)
        
        # Calculate Real-Time Spatial Distance profiles
        ego_loc = ego_vehicle.get_transform().location
        ped_loc = ped_actor.get_transform().location
        current_distance = ego_loc.distance(ped_loc)
        
        if current_distance < min_distance_to_target:
            min_distance_to_target = current_distance
            
        if current_distance < 2.0:
            collision_detected = True
            
        # Get Vehicle Velocity
        v_vector = ego_vehicle.get_velocity()
        current_speed_kmh = 3.6 * math.sqrt(v_vector.x**2 + v_vector.y**2 + v_vector.z**2)
        
        # Default Control Profile (Unassisted Baseline)
        target_throttle = 0.45
        target_brake = 0.0
        
        # 3. Process Ingested V2X Message Payloads
        incoming_broadcast = v2x_broker.receive_latest()
        
        if USE_V2X_ASSISTANCE and incoming_broadcast is not None:
            for obj in incoming_broadcast["detected_objects"]:
                target_pos = carla.Location(x=obj["world_x"], y=obj["world_y"], z=obj["world_z"])
                
                # Check if the detected pedestrian is in the path of our vehicle
                longitudinal_hazard_dist = target_pos.x - ego_loc.x
                lateral_deviation = abs(target_pos.y - ego_loc.y)
                
                # If a hazard is within our travel lane down the road
                if 0.0 < longitudinal_hazard_dist < 35.0 and lateral_deviation < 3.5:
                    if v2x_lead_time_discovered is None:
                        v2x_lead_time_discovered = time.time() - start_time
                        print(f"   [V2X ALERT] Hazard verified by Remote Node! Lead Time: {v2x_lead_time_discovered:.2f}s")
                    
                    # Proactive proportional slowing policy based on distance
                    if longitudinal_hazard_dist > 15.0:
                        target_throttle = 0.10
                        target_brake = 0.25  # Smooth preparation deceleration
                        world.debug.draw_string(ego_loc + carla.Location(z=2.5), "V2X: SLOWING", life_time=0.05, color=carla.Color(255, 165, 0))
                    else:
                        target_throttle = 0.0
                        target_brake = 1.0   # Controlled stop
                        world.debug.draw_string(ego_loc + carla.Location(z=2.5), "V2X: HAZARD STOP", life_time=0.05, color=carla.Color(255, 0, 0))
                        
        # Local Line-of-Sight Sensor Emulation (Ego Vehicle fallback perception)
        # Ego can only see the pedestrian without V2X once they clear the building wall edge
        if (ped_loc.y - ego_loc.y) < 4.0 and abs(ped_loc.x - ego_loc.x) < 12.0:
            if not USE_V2X_ASSISTANCE:
                world.debug.draw_string(ego_loc + carla.Location(z=2.5), "LOCAL SENSOR EMERGENCY BRAKE", life_time=0.05, color=carla.Color(255, 0, 0))
                target_throttle = 0.0
                target_brake = 1.0
                
        # Apply Computed Control Loop Commands
        ego_vehicle.apply_control(carla.VehicleControl(throttle=float(target_throttle), brake=float(target_brake), steer=0.0))
        
        # Render visual tracking markers within the simulator environment
        world.debug.draw_string(ped_loc + carla.Location(z=2.0), "PEDESTRIAN", life_time=0.05, color=carla.Color(0, 255, 255))
        
        # Dynamic telemetry updates on the command line
        if int(time.time() - start_time) % 2 == 0 and (time.time() - start_time) % 1.0 < 0.05:
            print(f"   Time={time.time()-start_time:.1f}s | Speed={current_speed_kmh:.1f} km/h | Target Dist={current_distance:.1f}m")
            
        time.sleep(0.03)

    # 4. Generate Performance Evaluation Summary Reports
    print("\n" + "="*50 + "\n VAL DELIVERABLES: SCENARIO PERFORMANCE METRICS\n" + "="*50)
    print(f" - Avoided Near Misses / Collision Occurred : {'CRASH DETECTED' if collision_detected else 'SUCCESSFUL AVOIDANCE'}")
    print(f" - Minimum Absolute Distance Recorded       : {min_distance_to_target:.2f} meters")
    print(f" - V2X Warning Lead Time Ingestion           : {f'{v2x_lead_time_discovered:.2f} seconds' if v2x_lead_time_discovered else 'N/A'}")
    print("="*50)

finally:
    # 5. Perform Environment Takedowns and Thread Safeties
    print(">> Stopping infrastructure processing threads and wiping active simulation actors...")
    network_stop_signal.set()
    infra_thread.join(timeout=2.0)
    safe_destroy(spawned_actors)
    print(">> Cleaned up.")